# Check and run carry the same refusal

Cancellations are temporarily paused for everyone. Compare the two public calls for the same manager and order.

Run with the Python environment of this checkout (`.venv`). These examples use the API in this branch; installing a released package may give a different API.

[Script](07_three_answers.py) · [Tutorial](../../docs/tutorials/step-03-authorization-and-roles.md)

## Imports

In [1]:
"""Check and run carry the same refusal."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.exceptions import AccessDenied
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## The rule

Cancellations are temporarily paused for everyone. Compare the two public calls for the same manager and order.

In [2]:
class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole, guard=lambda user, params: False, guard_reason="CANCELLATIONS_PAUSED")
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params, state, box, connections):
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)

## Try it

Run the cells in order. The calls below are the same as in the script.

In [3]:
machine = ActionProductMachine(cache_coordinator=None)
caller = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))

In [4]:
answer = await machine.check_access_decide(caller, CancelOrderAction, OrderParams(order_id="ord-001"))
print("check:", answer.model_dump(mode="json"))
try:
    await machine.run(caller, CancelOrderAction(), OrderParams(order_id="ord-001"))
except AccessDenied as exc:
    print("run raises:", type(exc).__name__)
    print("run verdict:", exc.verdict.model_dump(mode="json"))

check: {'kind': 'refused', 'gate': 'GUARD', 'reason': 'CANCELLATIONS_PAUSED'}
run raises: AccessDenied
run verdict: {'kind': 'refused', 'gate': 'GUARD', 'reason': 'CANCELLATIONS_PAUSED'}


## What the result means

The same refusal is returned by check_access_decide and carried by AccessDenied from run. Both evaluate the same policy. This comparison assumes the data does not change between calls.